# Callaway & Sant'Anna Group-Time DiD

**Econometrics Notebook Library · v0.1.0**

## Intuition

Do not begin with one regression coefficient. Begin with the primitive causal object:

$$ATT(g,t)=E[Y_t(1)-Y_t(0)\mid G=g].$$

For each treated cohort $g$ and calendar time $t$, construct a valid control group and estimate that group-time effect. Only then aggregate across $g$, $t$, or event time.

Callaway & Sant'Anna allow outcome-regression, inverse-probability-weighted, and doubly-robust estimands under conditional parallel trends.

Reference: [Callaway & Sant'Anna, Journal of Econometrics (2021)](https://doi.org/10.1016/j.jeconom.2020.12.001).

## Doubly-robust two-period score

For one $(g,t)$ comparison, use baseline $b=g-1$, define $\Delta Y=Y_t-Y_b$, and let $D=1\{G=g\}$. For a valid control pool, estimate the propensity $p(X)=P(D=1\mid X)$ and control outcome regression $m_0(X)=E[\Delta Y\mid X,D=0]$.

A panel doubly-robust ATT score is

$$
\psi_i = \frac{D_i(\Delta Y_i-m_0(X_i))}{E[D]}
-\frac{(1-D_i)\,p(X_i)}{(1-p(X_i))E[D]}
(\Delta Y_i-m_0(X_i)).
$$

Consistency survives misspecification of one nuisance model if the other is correctly specified, under the design assumptions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import simulate_staggered_panel, cs_att_gt, aggregate_att_event

df = simulate_staggered_panel(n_units=280, seed=41)
att_gt = cs_att_gt(df, use_covariate=True, control="not_yet")
att_gt.head(12)

In [ ]:
agg = aggregate_att_event(att_gt)
truth = (df[df.treated.eq(1)]
         .groupby("event_time", as_index=False)["tau_true"].mean()
         .rename(columns={"tau_true":"truth"}))
plot = agg.merge(truth, on="event_time", how="left")
plot.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(plot.event_time, plot.truth, marker="o", label="Truth")
ax.errorbar(plot.event_time, plot.estimate, yerr=1.96*plot.se, marker="o", capsize=3, label="CS DR aggregation")
ax.set(xlabel="Event time", ylabel="ATT", title="Estimate ATT(g,t) first; aggregate second")
ax.legend();

## Control-group choice is part of identification

`never-treated` controls are conceptually simple but may be unrepresentative. `not-yet-treated` controls use more observations but the composition changes with calendar time. Neither choice is “just a software option.” It changes which counterfactual trend restriction is being invoked.

In [ ]:
never = aggregate_att_event(cs_att_gt(df, use_covariate=True, control="never"))
comparison = agg.merge(never, on="event_time", suffixes=("_notyet","_never"))
comparison[["event_time","estimate_notyet","estimate_never"]].head(8)

## Researcher failure checklist

- Define $ATT(g,t)$ before choosing an aggregation.
- State whether controls are never-treated or not-yet-treated.
- Diagnose propensity-score overlap when using covariates.
- Do not aggregate group-time effects with arbitrary software defaults if cohort composition matters substantively.
- Separate conditional parallel trends from unconditional parallel trends in the paper.